# 02 · Baseline models

**Goal:** get a reference validation ROC-AUC using only the raw features,
and produce a first Kaggle submission. Every later improvement is measured against these numbers.

**Validation:** the test set contains *future* time steps (36–49), so we validate the same way:
train on labelled rows from time steps 1–28 and validate on time steps 29–35. No random splits, due to look-ahead data leakage issues.


**Models compared**
1. Logistic regression: the simplest reasonable model (the "true" baseline).
2. Random forest: works well with little tuning and handles irrelevant and correlated features well.
3. LightGBM: a gradient-boosted tree model, usually strong on tabular data.


In [1]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

SEED = 42
DATA_DIR = "../data"          # notebook lives in notebooks/, data/ is one level up
SUB_DIR = "../submissions"
os.makedirs(SUB_DIR, exist_ok=True)

## 1. Load data

`label` is stored as `1`, `0` or the text `none`. Converting to numeric turns `none` into `NaN`
(unlabelled). Only labelled rows can be used for supervised training.

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
train["label"] = pd.to_numeric(train["label"], errors="coerce")   # "none" -> NaN

feat_cols  = [f"feat_{i}" for i in range(1, 166)]   # all 165 features


lab = train[train["label"].notna()].copy()
lab["label"] = lab["label"].astype(int)

print(f"Train rows: {len(train):,} | labelled: {len(lab):,} | test rows: {len(test):,}")
print(f"Illicit rate among labelled: {lab['label'].mean():.2%}")

Train rows: 141,772 | labelled: 31,235 | test rows: 15,329
Illicit rate among labelled: 11.67%


## 2. Time-based train / validation split

In [3]:
VAL_START = 29   # validate on the last 7 training time steps (29–35)

tr = lab[lab["time_step"] <  VAL_START]
va = lab[lab["time_step"] >= VAL_START]

print(f"Train: steps 1–{VAL_START-1}, {len(tr):,} rows, {tr['label'].mean():.1%} illicit")
print(f"Val:   steps {VAL_START}–35, {len(va):,} rows, {va['label'].mean():.1%} illicit")

Train: steps 1–28, 25,207 rows, 10.1% illicit
Val:   steps 29–35, 6,028 rows, 18.3% illicit


## 3. Models

Settings are deliberately simple and close to defaults; tuning comes later.
- Logistic regression needs scaled inputs, so it is wrapped in a `StandardScaler` pipeline.
  `class_weight="balanced"` compensates for the ~12% illicit rate.
- Tree models don't need scaling. ROC-AUC is rank-based, so class imbalance matters less for it
  than for accuracy; we leave the trees unweighted for the baseline.

In [4]:
def make_models():
    return {
        "LogisticRegression": make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=500, max_features=50, n_jobs=-1, random_state=SEED,
        ),
        "LightGBM": lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.03, num_leaves=31,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.5,
            random_state=SEED, verbose=-1,
        ),
    }

## 4. Compare models on the validation period

In [5]:
results, val_preds = [], {}

for m_name, model in make_models().items():
    model.fit(tr[feat_cols], tr["label"])
    p = model.predict_proba(va[feat_cols])[:, 1]
    val_preds[m_name] = p
    auc = roc_auc_score(va["label"], p)
    results.append({"model": m_name, "val_AUC": round(auc, 4)})
    print(f"{m_name:18s} | val AUC = {auc:.4f}")

results = pd.DataFrame(results).sort_values("val_AUC", ascending=False).reset_index(drop=True)
results

LogisticRegression | val AUC = 0.8969
RandomForest       | val AUC = 0.9913
LightGBM           | val AUC = 0.9974


,model,val_AUC
0,LightGBM,0.9974
1,RandomForest,0.9913
2,LogisticRegression,0.8969


## 5. Stability over time: AUC per validation time step

A single overall AUC can hide a model that is great on some periods and poor on others.
Since the test set is further in the future, consistency across steps matters.

In [6]:
def auc_or_nan(y_true, y_pred):
    """ROC-AUC, or NaN if a time step contains only one class (AUC is undefined)."""
    return roc_auc_score(y_true, y_pred) if len(np.unique(y_true)) == 2 else np.nan

rows = []
for t in sorted(va["time_step"].unique()):
    mask = (va["time_step"] == t).to_numpy()
    y_t = va.loc[mask, "label"]
    row = {"time_step": t, "n_rows": int(mask.sum()), "n_illicit": int(y_t.sum())}
    for m_name, p in val_preds.items():
        row[m_name] = auc_or_nan(y_t, p[mask])
    rows.append(row)

per_step = pd.DataFrame(rows).set_index("time_step").round(3)
model_cols = list(val_preds)
per_step.loc["mean"]  = per_step[model_cols].mean().round(3)
per_step.loc["worst"] = per_step[model_cols].min().round(3)
per_step

,n_rows,n_illicit,LogisticRegression,RandomForest,LightGBM
time_step,,,,,
29,1174.0,329.0,0.935,0.990,0.997
30,524.0,83.0,0.961,0.988,0.995
31,710.0,106.0,0.810,0.989,0.992
32,1323.0,342.0,0.869,0.999,0.999
33,441.0,23.0,0.963,0.986,0.995
34,515.0,37.0,0.915,0.996,0.999
35,1341.0,182.0,0.942,0.995,0.997
mean,NaN,NaN,0.914,0.992,0.996
worst,NaN,NaN,0.810,0.986,0.992


## 6. First submission

Retrain the best model on **all** labelled rows (steps 1–35),
then predict the test set. More recent data is the most useful for predicting the future,
so we never leave the validation steps out of the final fit.

In [7]:
best = results.iloc[0]

print(f"Best on validation: {best['model']} (val AUC {best['val_AUC']})")

final_model = make_models()[best["model"]].fit(lab[feat_cols], lab["label"])

sub = pd.DataFrame({
    "index":  test["index"],
    "target": final_model.predict_proba(test[feat_cols])[:, 1],
})

# Sanity checks against the expected submission format
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
assert list(sub.columns) == list(sample.columns), "column names differ from sample_submission"
assert len(sub) == len(sample) and set(sub["index"]) == set(sample["index"]), "index mismatch"
assert sub["target"].between(0, 1).all()

out_path = f"{SUB_DIR}/01_baseline_{best['model']}.csv"
sub.to_csv(out_path, index=False)
print(f"Saved {out_path} ({len(sub):,} rows)")
sub.head()

Best on validation: LightGBM (val AUC 0.9974)
Saved ../submissions/01_baseline_LightGBM.csv (15,329 rows)


,index,target
0,0,0.000048
1,1,0.000368
2,2,0.000177
3,3,0.000067
4,4,0.000010


## 7. Experiment log

Copy these into your running experiment log (used later for the report):

| # | Experiment | Val AUC | Public LB |
|---|---|---|---|
| 1 | Baseline LightGBM  | *0.996* | *0.947* |